In [ ]:
# Load a real dataset (California Housing) directly from sklearn
from sklearn.datasets import fetch_california_housing
import pandas as pd

data = fetch_california_housing(as_frame=True)
df = data.frame  # already structured tabular data for ETL and ML

# Save locally for later use (Cloud Storage → BigQuery → ETL/ML pipeline)
df.to_csv("housing_sample.csv", index=False)

In [ ]:
!pip install scikit-learn==1.2.2 numpy==1.26.4 joblib --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 31.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 209.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 240.4 MB/s eta 0:00:00
  Created wheel for scikit-learn: filename=scikit_learn-1.2.2-cp312-cp312-linux_x86_64.whl size=9452393 sha256=f9a823f62531bfd998f43baf98ce4c778ddd2a63f0829af96e47908143cb9c4d
  Stored in directory: /tmp/pip-ephem-wheel-cache-av7w_j24/wheels/24/f8/77/ae90c181b806f450a6fec8c8f794594e7c92fa79d7ca27e656
Successfully built scikit-learn
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfull

In [ ]:
import sklearn, numpy
print(sklearn.__version__, numpy.__version__)

1.2.2 1.26.4


In [ ]:
# ============================================================
# Load libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.base import clone

# ============================================================
# Load dataset
# ============================================================
# Load the cleaned dataset exported from BigQuery
# Replace the file path if the CSV is stored in another location in Colab / Drive
file_path = "/content/housing_clean_v1.csv"
df = pd.read_csv(file_path)

# Display basic information to verify structure and schema
print("Shape:", df.shape)
display(df.head())
display(df.describe())

# ============================================================
# Define target and features
# ============================================================
# Define the prediction target
target_col = "MedHouseVal"

# Separate features from target
X = df.drop(columns=[target_col])
y = df[target_col]

# Detect numeric and categorical columns dynamically
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# ============================================================
# Engineer additional features
# ============================================================
# Create a working copy to add derived variables
X_fe = X.copy()

# Create domain-style engineered features to enrich the model input
if {"AveRooms", "AveBedrms"}.issubset(X_fe.columns):
    X_fe["RoomsPerBedroom"] = X_fe["AveRooms"] / (X_fe["AveBedrms"] + 1e-6)

if {"Population", "AveOccup"}.issubset(X_fe.columns):
    X_fe["EstimatedHouseholds"] = X_fe["Population"] / (X_fe["AveOccup"] + 1e-6)

if {"Latitude", "Longitude"}.issubset(X_fe.columns):
    X_fe["GeoInteraction"] = X_fe["Latitude"] * X_fe["Longitude"]

# Refresh feature lists after feature engineering
numeric_features = X_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_fe.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# ============================================================
# Split data
# ============================================================
# Split into train and test sets to evaluate generalization
X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, random_state=42
)

X_train = X_train.values
X_test = X_test.values

# ============================================================
# Build preprocessing
# ============================================================
# Prepare numeric transformation pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prepare categorical transformation pipeline if needed
# Use a safe fallback when no categorical columns exist

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, list(range(len(numeric_features))))
])


# ============================================================
# Define candidate models
# ============================================================
# Create multiple models to compare baseline and stronger learners
models = {
    "linear_regression": LinearRegression(),
    "ridge": Ridge(alpha=1.0),
    "random_forest": RandomForestRegressor(
        n_estimators=250,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    ),
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=300,
        min_samples_leaf=20,
        random_state=42
    )
}

# ============================================================
# Cross-validate all models
# ============================================================
# Use K-Fold CV to compare models with robust evaluation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "r2": "r2",
            "neg_mae": "neg_mean_absolute_error",
            "neg_rmse": "neg_root_mean_squared_error"
        },
        n_jobs=-1,
        return_train_score=False
    )

    results.append({
        "model": model_name,
        "cv_r2_mean": np.mean(scores["test_r2"]),
        "cv_mae_mean": -np.mean(scores["test_neg_mae"]),
        "cv_rmse_mean": -np.mean(scores["test_neg_rmse"])
    })

results_df = pd.DataFrame(results).sort_values(by="cv_r2_mean", ascending=False).reset_index(drop=True)
print("\nCross-validation results:")
display(results_df)

# ============================================================
# Select best base model
# ============================================================
best_model_name = results_df.iloc[0]["model"]
print("Best model from CV:", best_model_name)

best_base_model = models[best_model_name]

# ============================================================
# Tune the best tree-based model if applicable
# ============================================================
# Apply hyperparameter tuning to improve the selected model when useful
if best_model_name == "random_forest":
    param_dist = {
        "model__n_estimators": [200, 300, 400, 500],
        "model__max_depth": [8, 10, 12, 15, None],
        "model__min_samples_split": [2, 5, 10, 15],
        "model__min_samples_leaf": [1, 2, 4, 6],
        "model__max_features": ["sqrt", "log2", 0.8]
    }

    base_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
    ])

    tuner = RandomizedSearchCV(
        estimator=base_pipeline,
        param_distributions=param_dist,
        n_iter=20,
        scoring="r2",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    tuner.fit(X_train, y_train)
    final_model = tuner.best_estimator_
    print("Best tuned parameters:", tuner.best_params_)

elif best_model_name == "hist_gradient_boosting":
    param_dist = {
        "model__learning_rate": [0.03, 0.05, 0.08, 0.1],
        "model__max_depth": [4, 6, 8, 10],
        "model__max_iter": [200, 300, 400],
        "model__min_samples_leaf": [10, 20, 30, 50]
    }

    base_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    tuner = RandomizedSearchCV(
        estimator=base_pipeline,
        param_distributions=param_dist,
        n_iter=12,
        scoring="r2",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    tuner.fit(X_train, y_train)
    final_model = tuner.best_estimator_
    print("Best tuned parameters:", tuner.best_params_)

else:
    final_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", clone(best_base_model))
    ])
    final_model.fit(X_train, y_train)

# ============================================================
# Train final model
# ============================================================
# Train the final selected model on the training set
if not hasattr(final_model, "predict"):
    raise ValueError("Final model is not ready for prediction.")

if not hasattr(final_model, "named_steps"):
    raise ValueError("Final model is expected to be a sklearn Pipeline.")

# Fit only if the tuning branch did not already fit it
try:
    _ = final_model.predict(X_test[:1])
except:
    final_model.fit(X_train, y_train)

# ============================================================
# Evaluate final model
# ============================================================
# Generate predictions and calculate business-relevant regression metrics
y_pred = final_model.predict(X_test)

metrics = {
    "r2": r2_score(y_test, y_pred),
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred))
}

print("\nFinal model metrics on test set:")
for k, v in metrics.items():
    print(f"{k}: {v:.5f}")

# Create a comparison table for actual vs predicted values
predictions_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "absolute_error": np.abs(y_test.values - y_pred)
})

print("\nPrediction sample:")
display(predictions_df.head(20))

# ============================================================
# Extract feature importance when available
# ============================================================
# Try to compute feature importance for tree-based models
feature_importance_df = None

model_step = final_model.named_steps["model"]
preprocessor_step = final_model.named_steps["preprocessor"]

try:
    transformed_feature_names = preprocessor_step.get_feature_names_out()
except:
    transformed_feature_names = np.array([f"feature_{i}" for i in range(len(numeric_features))])

if hasattr(model_step, "feature_importances_"):
    feature_importance_df = pd.DataFrame({
        "feature": transformed_feature_names,
        "importance": model_step.feature_importances_
    }).sort_values(by="importance", ascending=False)

    print("\nTop feature importances:")
    display(feature_importance_df.head(20))

# ============================================================
# Save artifacts
# ============================================================
# Save trained model and evaluation outputs for later deployment in Vertex AI
os.makedirs("/content/artifacts", exist_ok=True)

joblib.dump(final_model, "/content/artifacts/housing_price_model.joblib")
predictions_df.to_csv("/content/artifacts/predictions_sample.csv", index=False)
results_df.to_csv("/content/artifacts/model_comparison_cv.csv", index=False)

with open("/content/artifacts/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

if feature_importance_df is not None:
    feature_importance_df.to_csv("/content/artifacts/feature_importance.csv", index=False)

print("\nArtifacts saved in /content/artifacts")
print(os.listdir("/content/artifacts"))
import sklearn, numpy
print("FINAL sklearn:", sklearn.__version__)
print("FINAL numpy:", numpy.__version__)

# ============================================================
# Create an inference example
# ============================================================
# Generate a single prediction example to document inference behavior
sample_record = X_test[[0]]
sample_prediction = final_model.predict(sample_record)[0]

print("\nSingle inference example:")
display(sample_record)
print("Predicted MedHouseVal:", round(float(sample_prediction), 5))

Shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,5.2636,1,7.694030,1.279851,872,3.253731,38.23,-122.00,1.913
1,4.2500,1,20.125000,2.928571,402,3.589286,37.65,-120.93,1.892
2,4.8750,1,5.533333,1.000000,32,2.133333,35.08,-117.95,1.417
3,1.6250,1,3.000000,1.000000,8,4.000000,33.86,-116.95,0.550
4,6.6410,2,5.483051,1.152542,203,1.720339,37.91,-122.51,3.100


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


Numeric features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Categorical features: []

Cross-validation results:


,model,cv_r2_mean,cv_mae_mean,cv_rmse_mean
0,hist_gradient_boosting,0.841804,0.305980,0.458597
1,random_forest,0.795420,0.346899,0.521586
2,linear_regression,0.644567,0.501184,0.687642
3,ridge,0.644513,0.501108,0.687694


Best model from CV: hist_gradient_boosting
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best tuned parameters: {'model__min_samples_leaf': 50, 'model__max_iter': 400, 'model__max_depth': 10, 'model__learning_rate': 0.08}

Final model metrics on test set:
r2: 0.84800
mae: 0.29350
rmse: 0.45031

Prediction sample:


,actual,predicted,absolute_error
0,5.00001,3.572764,1.427246
1,0.91900,0.731950,0.187050
2,0.81500,0.945161,0.130161
3,1.62500,2.015579,0.390579
4,2.63100,2.856293,0.225293
5,1.66800,1.668148,0.000148
6,1.36900,1.509944,0.140944
7,2.65700,2.556545,0.100455
8,0.58100,0.625785,0.044785
9,4.80800,2.776418,2.031582



Artifacts saved in /content/artifacts
['test_metrics.json', 'housing_price_model.joblib', 'predictions_sample.csv', 'model_comparison_cv.csv']
FINAL sklearn: 1.2.2
FINAL numpy: 1.26.4

Single inference example:


array([[ 6.15750000e+00,  5.20000000e+01,  7.57333333e+00,
         1.21333333e+00,  2.19000000e+02,  2.92000000e+00,
         3.74500000e+01, -1.22150000e+02,  6.24175310e+00,
         7.49999743e+01, -4.57451750e+03]])

Predicted MedHouseVal: 3.57276
